In [1]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination
import sys
import os
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

Speaker Selection:

Instead of round-robin or manual control, a model-based selection function chooses the next agent to speak.

This ensures context-aware collaboration, where the most relevant agent responds.

Broadcasting Messages:

When an agent speaks, its message is broadcast to all other participants.

This keeps the conversation synchronized across the team.

Configurable Roles:

Each agent can have a role and description (e.g., “Data Analyst,” “Summarizer,” “Critic”).

Roles guide the model in deciding who should speak next.

Turn Management:

Optionally prevents the same agent from speaking twice in a row.

Helps maintain diversity in contributions.

Customizable Selection Logic:

You can override the default model-based selection with your own function.

You can also define a candidate function to narrow down which agents are eligible to speak.

In [2]:
model_client = OpenAIChatCompletionClient(
    model='gpt-3.5-turbo'
)

In [3]:
async def location_planner(location: str) -> str:
    return f"Here are the places to visit in {location}: loc1, loc2, loc3"

In [4]:
async def hotel_selection(location: str) -> str:
    return f"Here are the good hotels near {location}: hotel1, hotel2, hotel3"

In [5]:
async def flight_selection(origin:str, destination:str)->str:
    return f"Here are the flights from {origin} to {destination}: flight1, flight2, flight3"

In [6]:
travel_planner = AssistantAgent(
    model_client= model_client,
    name="Travel_Planner",
    description="A travel planner who helps in planning the travel itinerary",
    tools=[location_planner]
)

In [7]:
hotel_planner = AssistantAgent(
    name="Hotel_Planner",
    description="Helps in identifying the good hotels near the visiting location",
    model_client= model_client,
    tools=[hotel_selection]
)

In [8]:
flight_planner = AssistantAgent(
    name = "Flight_Planner",
    description="Helps in identifying various flights from origin to destination",
    model_client= model_client,
    tools=[flight_selection]
)

In [9]:
termination = TextMentionTermination("exit")

In [12]:
team = SelectorGroupChat(
    [flight_planner, travel_planner, hotel_planner],
    model_client= model_client,
    termination_condition=termination,
    max_turns=3
)

In [14]:
await Console(team.run_stream(task = "Book a three day flight trip from Washington to New York"))

---------- TextMessage (user) ----------
Book a three day flight trip from Washington to New York
---------- ToolCallRequestEvent (Travel_Planner) ----------
[FunctionCall(id='call_BI5q9942o5EBkjox7aC45Wnu', arguments='{"location": "New York"}', name='location_planner'), FunctionCall(id='call_ARsDDG4wOWbsVRuCzeqJjtrX', arguments='{"location": "Washington"}', name='location_planner')]
---------- ToolCallExecutionEvent (Travel_Planner) ----------
[FunctionExecutionResult(content='Here are the places to visit in New York: loc1, loc2, loc3', name='location_planner', call_id='call_BI5q9942o5EBkjox7aC45Wnu', is_error=False), FunctionExecutionResult(content='Here are the places to visit in Washington: loc1, loc2, loc3', name='location_planner', call_id='call_ARsDDG4wOWbsVRuCzeqJjtrX', is_error=False)]
---------- ToolCallSummaryMessage (Travel_Planner) ----------
Here are the places to visit in New York: loc1, loc2, loc3
Here are the places to visit in Washington: loc1, loc2, loc3
---------- T

TaskResult(messages=[TextMessage(id='38a10439-fe6c-4528-8853-b5dbbd26d750', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 9, 7, 32, 26, 470782, tzinfo=datetime.timezone.utc), content='Book a three day flight trip from Washington to New York', type='TextMessage'), ToolCallRequestEvent(id='606cfea1-e1f9-419c-b399-33b6b80434db', source='Travel_Planner', models_usage=RequestUsage(prompt_tokens=525, completion_tokens=46), metadata={}, created_at=datetime.datetime(2026, 4, 9, 7, 32, 28, 561675, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_BI5q9942o5EBkjox7aC45Wnu', arguments='{"location": "New York"}', name='location_planner'), FunctionCall(id='call_ARsDDG4wOWbsVRuCzeqJjtrX', arguments='{"location": "Washington"}', name='location_planner')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='bac795c8-af72-4a85-90f8-53c8a14a2fdd', source='Travel_Planner', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 9, 7, 3